In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#teacher mlp
input_dim = 32
batch_size = 256
val_size = 16384
torch.manual_seed(4948497)
teacher = nn.Sequential(nn.Linear(input_dim, 256), nn.GELU(), nn.Linear(256,256), nn.GELU(), nn.Linear(256,1))

torch.manual_seed(0)                                   # Teacher
teacher = nn.Sequential(nn.Linear(input_dim, 256), nn.GELU(), nn.Linear(256,256), nn.GELU(), nn.Linear(256,1))
for p in teacher.parameters():
    p.requires_grad_(False)

g_probe = torch.Generator().manual_seed(1)             # Probe: seed + 1
with torch.no_grad():
    out = teacher(torch.randn(65536, input_dim, generator=g_probe))
out_mean = out.mean().item()
out_scale = 1.0 / out.std().item()

def clean_target(x):
    with torch.no_grad():
        return (teacher(x) - out_mean) * out_scale
    
def cosine_lr_lambda(total_steps: int, warmup_frac: float = 0.08,
                     start_factor: float = 0.01, end_factor: float = 0.1):
    """LR multiplier: linear warmup eta/100 -> eta over ``warmup_frac`` of the steps,
    then cosine annealing eta -> eta/10. The per-cell cosine schedule used throughout."""
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def fn(step: int) -> float:
        if step < warmup_steps:
            return start_factor + (1.0 - start_factor) * step / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return end_factor + (1.0 - end_factor) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return fn

In [ ]:
def make_std(seed=384929):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(input_dim, 64), nn.GELU(), nn.Linear(64,64), nn.GELU(), nn.Linear(64,1))
'''
def train_student(lr, n_train_steps, seed=384929):
    mlp = make_std(seed)
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr_lambda(n_train_steps))
    
    for i in range(n_train_steps):
        x_input = torch.randn(batch_size, input_dim) # in oder outside of loop?
        with torch.no_grad():
            target = clean_target(x_input)

        optimizer.zero_grad()
        loss = loss_function(mlp(x_input),target)
        loss.backward()
        optimizer.step()
        scheduler.step()

    gv = torch.Generator().manual_seed(496302)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    with torch.no_grad():
        y_val = clean_target(x_val)

    val_loss = loss_function(mlp(x_val), y_val)
    return val_loss
'''
noise_std = 0.1

def train_student(lr, n_train_steps, seed=0):
    mlp = make_std(seed)
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr_lambda(n_train_steps))

    g = torch.Generator().manual_seed(seed * 100003 + 64)   # wie im Referenzcode
    for i in range(n_train_steps):
        x_input = torch.randn(batch_size, input_dim, generator=g)
        target = clean_target(x_input) + noise_std * torch.randn(batch_size, 1, generator=g)

#        x_input = torch.randn(batch_size, input_dim)
 #       target = clean_target(x_input)
        
        optimizer.zero_grad()
        loss = loss_function(mlp(x_input), target)
        loss.backward()
        optimizer.step()
        scheduler.step()

    gv = torch.Generator().manual_seed(2)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    with torch.no_grad():
        y_val = clean_target(x_val)
        val_loss = loss_function(mlp(x_val), y_val) + noise_std ** 2   # + sigma^2
    return val_loss

In [ ]:
#ranges = [(1e-5, (3e-4, 1e-1), (1e-3, 1e-1), (1e-2, 1e-1)]   # Beispielwerte, anpassen

ranges = [ (3e-4, 1e-1)]        #(1e-5, 8e-1), (1e-4, 5e-1),
n_seeds = 5

learn_rates_list, means, stds = [], [], []
for lo, hi in ranges:
    lrs = np.geomspace(lo, hi, 9)
    m, s = [], []
    for lr in lrs:
        vals = [train_student(lr, 2048, seed=seed).item() for seed in range(n_seeds)]
        m.append(np.mean(vals))
        s.append(np.std(vals))
    learn_rates_list.append(lrs)
    means.append(np.array(m))
    stds.append(np.array(s))
    

In [ ]:
def plot_lr_tuning(lrs_list, means_list, stds_list, ranges, figsize=(10, 6)):
    #für gestackte plots brauchen wir dann wieder figsize=(5,7)
    """Gestapelte Figuren: validation loss vs learning rate mit Fehlerbalken (Std über Seeds)."""
    n = len(lrs_list)
    fig, axes = plt.subplots(n, 1, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, lrs, losses, stds, (lo, hi) in zip(axes, lrs_list, means_list, stds_list, ranges):
        lrs = np.asarray(lrs, dtype=float)
        #x = np.log10(lrs)
        #a, b, c = np.polyfit(x, losses, 2)
        #xg = np.linspace(x.min(), x.max(), 100)
        #ax.plot(10 ** xg, a * xg ** 2 + b * xg + c, "-", color="0.6", lw=1.5, label="parabola fit")
        ax.errorbar(lrs, losses, yerr=stds, fmt="o", color="steelblue", ms=7,
                    capsize=3, label="mean ± std")
        ax.set(xscale="log", ylabel="validation loss", title=f"{lo:.0e} bis {hi:.0e}")
        ax.legend(frameon=False)
    axes[-1].set_xlabel(r"learning rate $\eta$")

    fig.tight_layout(rect=[0, 0, 1, 0.96])   # oberen 4 % für den Supertitle freihalten

    info = []
    info.append(f"{2048} training steps")
    info.append(f"{n_seeds} seeds")
    if info:
        fig.suptitle(", ".join(info), fontsize=10)


plot_lr_tuning(learn_rates_list, means, stds, ranges)

In [ ]:
# ===== TEMPORÄR: Vergleich alt (ohne Rauschen) vs. neu (mit Rauschen) =====
# Anpassen: so setzen, wie du es für die neuen means/stds verwendet hast
seeds_cmp = range(5)      # <- gleiche Seeds wie bei deinen vorhandenen means/stds
steps_cmp = 2048 # <- gleiche Schrittzahl wie bei der neuen Variante

def train_student_old(lr, n_train_steps, seed=0):
    mlp = make_std(seed)
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr_lambda(n_train_steps))
    for i in range(n_train_steps):
        x_input = torch.randn(batch_size, input_dim)
        with torch.no_grad():
            target = clean_target(x_input)
        optimizer.zero_grad()
        loss = loss_function(mlp(x_input), target)
        loss.backward()
        optimizer.step()
        scheduler.step()
    gv = torch.Generator().manual_seed(496302)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    with torch.no_grad():
        y_val = clean_target(x_val)
        return float(loss_function(mlp(x_val), y_val))

means_old, stds_old = [], []
for lrs in learn_rates_list:
    vals = np.array([[train_student_old(lr, steps_cmp, seed=s) for s in seeds_cmp] for lr in lrs])
    means_old.append(vals.mean(axis=1))
    stds_old.append(vals.std(axis=1))

fig, axes = plt.subplots(len(learn_rates_list), 1, figsize=(10, 6))
axes = np.atleast_1d(axes)
for i, (ax, lrs, (lo, hi)) in enumerate(zip(axes, learn_rates_list, ranges)):
    lrs = np.asarray(lrs, dtype=float)
    ax.errorbar(lrs, means[i], yerr=stds[i], fmt="o", color="steelblue", ms=7,
                capsize=3, label="neu (mit Rauschen, + σ²)")
    ax.errorbar(lrs, means_old[i], yerr=stds_old[i], fmt="s", color="darkorange", ms=7,
                capsize=3, label="alt (ohne Rauschen)")
    ax.set(xscale="log", ylabel="validation loss", title=f"{lo:.0e} bis {hi:.0e}")
    ax.legend(frameon=False)
axes[-1].set_xlabel(r"learning rate $\eta$")
fig.tight_layout()
# ===== ENDE TEMPORÄR =====

In [ ]:
# ===== TEMPORÄR: beide Diagramme übereinanderlegen =====
import numpy as np
import matplotlib.pyplot as plt

lrs_cmp = np.geomspace(3e-4, 1e-1, 9)

# aus den Bildern abgelesen (linkes bzw. rechtes Diagramm)
loss_links  = [0.0339, 0.0304, 0.0289, 0.0272, 0.0255, 0.02545, 0.0274, 0.0345, 0.0374]
loss_rechts = [0.0349, 0.0306, 0.0291, 0.0276, 0.0265, 0.0271,  0.0295, 0.0372, 0.0392]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lrs_cmp, loss_links,  "o", color="steelblue", ms=7, label="without target noise")
ax.plot(lrs_cmp, loss_rechts, "o", color="red",       ms=7, label="with target noise")
ax.set(xscale="log", xlabel=r"learning rate $\eta$", ylabel="validation loss",
       title="3e-04 bis 1e-01")
ax.legend(frameon=False)
fig.tight_layout()
# ===== ENDE TEMPORÄR =====